In [71]:
import os
from glob import glob
from subprocess import run
from io import StringIO

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

os.chdir('/home/vladimirnoz/Projects/Codebook_Perspectives/')

In [72]:
# number of matrices

len(glob('motifs/manual_motifs_v2/pwms/*.pwm'))

235

In [73]:
# number of peaks
peaks = glob('peaks/chipseq+ghtselex/*.bed')
tfs_with_peaks = set([x.split('/')[-1].split('.')[0] for x in peaks])
len(tfs_with_peaks)

216

In [74]:
# number of TOPs

len(glob('TOPs/TOPs/*.bed'))

136

In [75]:
reads = glob('/home/vladimirnoz/AS_GHT_SELEX/chipseq-as/mapped_filtered_*/*.bam') + glob('/home/vladimirnoz/AS_GHT_SELEX/affiseq-as/mapped_filtered/*')
tfs_with_reads = set([x.split('/')[-1].split('.')[0] for x in reads])
len(tfs_with_reads)

207

In [76]:
'ZNF705EP' in tfs_with_reads

True

# ASB Stats

In [77]:
# snv unique positions

snps = glob('AS_CHS_GHTS/as_tables/*/*/*.tsv')
ids = set()
for file in snps:
    table = pd.read_table(file)
    ids = ids | set(table['id'])
len(ids)

122364

In [78]:
# snv unique positions with asb

snps = glob('AS_CHS_GHTS/as_tables/*/*/*.tsv')
ids = set()
for file in snps:
    table = pd.read_table(file).query('fdr_comb_pval < 0.05')
    ids = ids | set(table['id'])
len(ids)

10009

In [79]:
# chip-seq

#num_snps
snps = glob('AS_CHS_GHTS/SNPs_filtered/chipseq/*/*.vcf.gz')
total_snps = 0
for path in tqdm(snps):
    cmd = f'bcftools view -H {path} | wc -l'
    output = run(cmd, shell=True, text=True, capture_output=True).stdout
    num = int(output.strip())
    total_snps += num
print(total_snps)

  0%|          | 0/361 [00:00<?, ?it/s]

889814


In [80]:
# chip-seq

#num_tfs
tfs = glob('AS_CHS_GHTS/SNPs_filtered/chipseq/*')
len(tfs)

155

In [83]:
# chip-seq stats

datasets = glob('AS_CHS_GHTS/as_tables/chipseq/ASB_motifs/*.tsv')
total_asb = 0
pwm_hits = 0
concordant_hits = 0
strongly_concordant = 0
tfs = set()

for path in tqdm(datasets):
    df = pd.read_table(path).query('fdr_comb_pval < 0.05')
    if len(df) == 0:
        continue
    total_asb += len(df)
    tfs.add(path.split('/')[-1].split('.')[0])

    pwm_hits += len(df.query('motif_conc != "No Hit"'))
    concordant_hits += len(df.query('motif_conc == "Concordant"'))
    strongly_concordant += len(df.query('motif_conc == "Concordant" & abs(motif_fc) >= 1'))

    
print(len(tfs))
print(total_asb)
print(pwm_hits, f'{pwm_hits/total_asb:.0%}')
print(concordant_hits, f'{concordant_hits/pwm_hits:.0%}')
print(strongly_concordant)

  0%|          | 0/155 [00:00<?, ?it/s]

134
10575
2809 27%
1897 68%
1373


In [8]:
# ght-selex

#num_snps
snps = glob('AS_CHS_GHTS/SNPs_filtered/selex/*/*.vcf.gz')
total_snps = 0
for path in tqdm(snps):
    cmd = f'bcftools view -H {path} | wc -l'
    output = run(cmd, shell=True, text=True, capture_output=True).stdout
    num = int(output.strip())
    total_snps += num
print(total_snps)

  0%|          | 0/1303 [00:00<?, ?it/s]

35183


In [9]:
# ght-selex

#num_tfs
tfs = glob('AS_CHS_GHTS/SNPs_filtered/selex/*')
len(tfs)

178

In [10]:
# ght-selex

#num_experiments
datasets = glob('AS_CHS_GHTS/SNPs_filtered/selex/*/*')
len(set([x.split('.')[1] for x in datasets]))

370

In [84]:
# ght-selex stats

datasets = glob('AS_CHS_GHTS/as_tables/selex/ASB_motifs/*.tsv')
total_asb = 0
pwm_hits = 0
concordant_hits = 0
strongly_concordant = 0
tfs = set()

for path in tqdm(datasets):
    df = pd.read_table(path).query('fdr_comb_pval < 0.05')
    if len(df) == 0:
        continue
    total_asb += len(df)
    tfs.add(path.split('/')[-1].split('.')[0])

    pwm_hits += len(df.query('motif_conc != "No Hit"'))
    concordant_hits += len(df.query('motif_conc == "Concordant"'))
    strongly_concordant += len(df.query('motif_conc == "Concordant" & abs(motif_fc) >= 1'))

    
print(len(tfs))
print(total_asb)
print(pwm_hits, f'{pwm_hits/total_asb:.0%}')
print(concordant_hits, f'{concordant_hits/pwm_hits:.0%}')
print(strongly_concordant)

  0%|          | 0/178 [00:00<?, ?it/s]

162
1485
755 51%
469 62%
309


In [12]:
# Total ASBs

datasets = glob('AS_CHS_GHTS/as_tables/*/ASB_motifs/*.tsv')
total_asb = 0
pwm_hits = 0
concordant_hits = 0

for path in tqdm(datasets):
    df = pd.read_table(path).query('fdr_comb_pval < 0.05')
    if len(df) == 0:
        continue
    total_asb += len(df)

    pwm_hits += len(df.query('motif_conc != "No Hit"'))
    concordant_hits += len(df.query('motif_conc == "Concordant"'))

    

print(total_asb)
print(pwm_hits, f'{pwm_hits/total_asb:.0%}')
print(concordant_hits, f'{concordant_hits/pwm_hits:.0%}')

  0%|          | 0/333 [00:00<?, ?it/s]

12060
3564 30%
2366 66%


In [13]:
# ASBs in TOPs

datasets = glob('AS_CHS_GHTS/as_tables/*/ASB_motifs/*.tsv')
asbs_in_tops = 0


for path in tqdm(datasets):
    df = pd.read_table(path).query('fdr_comb_pval < 0.05')
    tf = os.path.basename(path).replace('.tsv', '')
    top_path = f'TOPs/TOPs/{tf}.bed'
    cmd = f'bedtools intersect -a stdin -b {top_path}'
    df_text = df.to_csv(index=False, sep='\t')
    proc = run(cmd, shell=True, text=True, capture_output=True, input=df_text)
    output = proc.stdout
    asbs_in_tops += len(output.split('\n'))
asbs_in_tops

  0%|          | 0/333 [00:00<?, ?it/s]

858

# Joint Collection & Remake

In [86]:
from pathlib import Path
import os

import polars as pl

import pandas as pd
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font
from copy import copy


os.chdir('/home/vladimirnoz/Projects/Codebook_Perspectives/AS_CHS_GHTS/')

TF_TABLES_PATH = Path('as_tables/')
JOINT_PATH = Path('joint_tables/ASB_Phenotypes/selex+chipseq.tsv')
TF_TYPES_PATH = Path('../common/tftypes.txt')
OUTPUT_DIR_PATH = Path('collection/')
FDR_THR = 0.05


fdr_filter = (pl.col('fdr_comb_pval') < 0.05)

In [87]:
types = pl.read_csv(TF_TYPES_PATH, separator='\t', has_header=False, new_columns=['tf', 'type'])
types

tf,type
str,str
"""KDM5B""","""Codebook TF"""
"""ARID2""","""Codebook TF"""
"""AHCTF1""","""Codebook TF"""
"""AHDC1""","""Codebook TF"""
"""AKNA""","""Codebook TF"""
…,…
"""RORB""","""Control"""
"""RXRA""","""Control"""
"""VDR""","""Control"""


In [96]:
tables_list = []

for path in TF_TABLES_PATH.glob('*/ASB_motifs/*.tsv'):
    tf, exp = path.stem.replace('ZNF705EP', 'ZNF705E'), path.parent.parent.stem
    local_df = pl.read_csv(path, separator='\t', infer_schema_length=10000)
    local_df = local_df.filter(fdr_filter).with_columns(
        pl.lit(tf).alias('tf'),
        pl.lit(exp).alias('experiment_type'),
    )
    tables_list.append(local_df)

tf_table = pl.concat(tables_list, how='vertical_relaxed')

In [97]:
tf_table

#chr,start,end,mean_bad,id,max_cover,ref,alt,n_reps,bads,scorefiles,ref_counts,alt_counts,ref_es,alt_es,ref_pval,alt_pval,ref_comb_es,alt_comb_es,ref_comb_pval,alt_comb_pval,ref_fdr_comb_pval,alt_fdr_comb_pval,pref_allele,comb_es,comb_pval,fdr_comb_pval,ref_motif_pos,ref_motif_orient,alt_motif_pos,alt_motif_orient,ref_motif_pval,alt_motif_pval,motif_fc,motif_conc,tf,experiment_type
str,i64,i64,f64,str,i64,str,str,i64,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,i64,str,i64,str,f64,f64,f64,str,str,str
"""chr19""",36128053,36128054,4.0,"""rs3810445""",67,"""G""","""C""",1,"""4""","""ZNF551/ZNF551.THC_0767.vcf.gz""","""61""","""6""","""2.6720383497220896""","""-4.183276479941598""","""0.00026279395076319207""","""0.9962656378061762""",2.672038,-4.183276,0.000263,0.996266,0.049259,1.0,"""ref""",2.672038,0.000263,0.049259,-11,"""direct""",-11,"""direct""",0.066678,0.008591,2.956331,"""No Hit""","""ZNF551""","""chipseq"""
"""chr17""",50909778,50909779,2.0,"""rs879778""",196,"""C""","""T""",2,"""2,2""","""ZNF551/ZNF551.THC_0671.vcf.gz,…","""100,137""","""28,59""","""2.0680071091887555,1.551172326…","""-1.4261561948964543,-0.8194090…","""0.0006459240582370776,0.024122…","""0.988551249414582,0.7049833361…",1.894104,-0.838757,0.000187,0.979246,0.044663,1.0,"""ref""",1.894104,0.000187,0.044663,-8,"""direct""",-20,"""direct""",0.00122,0.001635,-0.422304,"""No Hit""","""ZNF551""","""chipseq"""
"""chr15""",71307911,71307912,1.0,"""rs11633212""",389,"""G""","""A""",2,"""1,1""","""ZNF551/ZNF551.THC_0671.vcf.gz,…","""107,185""","""154,204""","""-0.4099995458017647,-0.0257238…","""0.6927085450408885,0.303013172…","""0.9766088056662167,0.559426850…","""0.0003716156867465743,0.035097…",-0.04077,0.576651,0.941636,0.000164,1.0,0.030741,"""alt""",0.576651,0.000164,0.030741,-1,"""revcomp""",-1,"""revcomp""",0.020675,0.009945,1.055859,"""No Hit""","""ZNF551""","""chipseq"""
"""chr10""",46422792,46422793,2.0,"""rs1105775""",762,"""C""","""T""",2,"""2,2""","""ZNF551/ZNF551.THC_0671.vcf.gz,…","""384,557""","""125,205""","""1.9576629072224163,1.767015098…","""-1.232605737198389,-1.05548093…","""3.348075208657172e-06,1.246592…","""0.9992972091927268,0.994496884…",1.867582,-1.075497,4.3169e-8,0.999938,0.000036,1.0,"""ref""",1.867582,4.3169e-8,0.000036,-13,"""direct""",-13,"""direct""",0.005814,0.007792,-0.422372,"""No Hit""","""ZNF551""","""chipseq"""
"""chr19""",39533621,39533622,4.0,"""rs11879944""",1095,"""A""","""G""",2,"""4,4""","""ZNF551/ZNF551.THC_0671.vcf.gz,…","""21,46""","""535,1049""","""-4.366878853064198,-4.20655993…","""3.685080767711961,3.6559051456…","""1.0,1.0""","""6.761214265029219e-25,2.481755…",-4.286719,3.666626,1.0,5.8633e-19,1.0,9.8913e-16,"""alt""",3.666626,5.8633e-19,9.8913e-16,-1,"""revcomp""",-1,"""revcomp""",0.008591,0.01786,-1.055821,"""No Hit""","""ZNF551""","""chipseq"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr12""",77444268,77444269,2.0,"""rs74105065""",120,"""G""","""T""",5,"""2,2,2,2,2""","""PAX7/PAX7.YWK_B_AffSeq_A10_PAX…","""57,92,63,29,20""","""23,28,14,31,15""","""1.2480968845811633,1.793440460…","""-1.235715492650713,-1.69597100…","""0.06730347774204186,0.00145191…","""0.7268151618278311,0.963819542…",1.52994,-0.378066,0.000193,0.969628,0.007045,1.0,"""ref""",1.52994,0.000193,0.007045,-9,"""direct""",-3,"""revcomp""",0.037128,0.018751,0.98552,"""No Hit""","""PAX7""","""selex"""
"""chr13""",68370895,68370896,2.0,"""rs7993555""",75,"""A""","""G""",3,"""2,2,2""","""PAX7/PAX7.YWK_B_AffSeq_A10_PAX…","""9,14,6""","""28,23,69""","""-1.5601964940311905,-0.7774382…","""1.0003961502861665,0.410909513…","""0.869674128690127,0.5531240879…","""0.13813202875864103,0.26450532…",-0.927018,2.120269,0.999558,0.00039,0.999563,0.00475,"""alt""",2.120269,0.00039,0.00475,-6,"""direct""",-6,"""revcomp""",0.003935,0.007421,-0.915332,"""No Hit""","""PAX7""","""selex"""
"""chr15""",73125484,73125485,2.0,"""rs8026579""",399,"""A""","""G""",5,"""2,2,2,2,2""","""PAX7/PAX7.YWK_B_Af

In [98]:
#unique tfs by types
tf_table_df = tf_table.lazy()
tf_table_df = tf_table_df.join(types.lazy(), on=['tf'])
tf_table_df.group_by(
    ['type']
).agg(
    pl.col('tf').n_unique()
).collect()

type,tf
str,u32
"""Control""",39
"""Codebook TF""",152


In [90]:
#unique signif SNPs

tf_table.group_by(
    ['#chr', 'end', 'id']
).len().select(pl.len())

len
u32
10009


In [91]:
#unique signif SNPs

tf_table.group_by(
    ['#chr', 'end', 'id', 'experiment_type']
).len().group_by(
    'experiment_type'    
).len()

experiment_type,len
str,u32
"""chipseq""",8750
"""selex""",1338


In [92]:
table = pl.read_csv(JOINT_PATH, separator='\t', infer_schema_length=10000).filter(fdr_filter)
table = table.drop(pl.col('ADASTRA_CL'))
print(len(table))

22064


In [93]:
table

#chr,start,end,mean_bad,id,max_cover,ref,alt,n_reps,ref_comb_es,alt_comb_es,ref_comb_pval,alt_comb_pval,ref_fdr_comb_pval,alt_fdr_comb_pval,pref_allele,comb_es,comb_pval,fdr_comb_pval,ebi,eQTL_cis,eQTL_cis_gene,ADASTRA_TF
str,i64,i64,f64,str,i64,str,str,i64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,str,str,str,str
"""chr2""",109614114,109614115,1.0,"""rs7579996""",334,"""C""","""A""",54,-0.031328,0.287821,0.352738,0.000014,0.67657,0.00055,"""alt""",0.287821,0.000014,0.00055,"""-""","""Adipose_Subcutaneous;Adipose_V…","""ENSG00000186522.14;ENSG0000019…","""ZN217_HUMAN;PRDM4_HUMAN;CTBP1_…"
"""chr7""",157010642,157010643,2.0,"""rs885832""",623,"""G""","""C""",95,1.02913,-0.876689,1.8214e-29,0.999976,2.7515e-26,1.0,"""ref""",1.02913,1.8214e-29,2.7515e-26,"""-""","""Esophagus_Mucosa;Skin_Sun_Expo…","""ENSG00000130675.14""","""ZN382_HUMAN;VEZF1_HUMAN;ZN341_…"
"""chr16""",1984135,1984136,2.0,"""rs56119017""",229,"""C""","""T""",89,1.282087,-1.198829,1.1817e-46,1.0,5.3555e-43,1.0,"""ref""",1.282087,1.1817e-46,5.3555e-43,"""-""","""Adipose_Subcutaneous;Adipose_V…","""ENSG00000127554.12;ENSG0000012…","""ETS1_HUMAN;ZBT7A_HUMAN;EP300_H…"
"""chr16""",15787933,15787934,2.0,"""rs9924070""",61,"""G""","""A""",3,-1.494518,1.597056,0.973591,0.001702,1.0,0.028244,"""alt""",1.597056,0.001702,0.028244,"""-""","""Adipose_Subcutaneous;Adipose_V…","""ENSG00000166780.10;ENSG0000017…","""ZN362_HUMAN;ZN384_HUMAN;RBM25_…"
"""chr1""",28648888,28648889,2.0,"""rs9426296""",214,"""G""","""T""",101,-0.897635,0.975315,0.999954,6.7478e-25,1.0,6.7129e-22,"""alt""",0.975315,6.7478e-25,6.7129e-22,"""-""","""Adipose_Subcutaneous;Adipose_V…","""ENSG00000130766.4;ENSG00000180…","""FOXM1_HUMAN;ARNT_HUMAN;INT13_H…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr9""",128662027,128662028,3.0,"""rs147818325""",86,"""G""","""A""",2,-2.439635,1.842131,0.931414,0.002337,1.0,0.035716,"""alt""",1.842131,0.002337,0.035716,"""-""","""-""","""-""","""-"""
"""chr11""",8068451,8068452,3.0,"""rs7114039""",97,"""T""","""C""",2,1.28489,-1.785032,0.000316,0.959219,0.007389,1.0,"""ref""",1.28489,0.000316,0.007389,"""-""","""Colon_Sigmoid;Colon_Transverse…","""ENSG00000166402.8""","""ARI1B_HUMAN;GCR_HUMAN;ANDR_HUM…"
"""chr12""",132472506,132472507,3.0,"""rs1574245""",498,"""G""","""A""",3,-1.811847,1.595495,0.645843,9.7295e-7,0.888833,0.000053,"""alt""",1.595495,9.7295e-7,0.000053,"""-""","""Brain_Anterior_cingulate_corte…","""ENSG00000112787.12;ENSG0000017…","""SSRP1_HUMAN"""


# Chromatin

In [14]:
# total datasets

!ls Chromatin/as_tables/ | wc -l

211030


In [15]:
# motif-annotated sets of ASchromatin sites

len(pd.read_table('Chromatin/agg_dnase/pvalues_init.tsv')) + len(pd.read_table('Chromatin/agg_atac/pvalues_init.tsv'))

7070

In [16]:
len(pd.read_table('Chromatin/agg_atac/pvalues_agg_greater.tsv'))

219

In [17]:
len(pd.read_table('Chromatin/agg_atac/pvalues_agg_greater.tsv').query('fdr_comb < 0.05'))

20

In [18]:
len(pd.read_table('Chromatin/agg_dnase/pvalues_agg_greater.tsv'))

215

In [19]:
len(pd.read_table('Chromatin/agg_dnase/pvalues_agg_greater.tsv').query('fdr_comb < 0.05'))

48

In [12]:
# unique tfs across dnase and atac
tfs = set()
for file in glob('Chromatin/agg_*/pvalues_agg_greater.tsv'):
    table = pd.read_table(file).query('fdr_comb < 0.05')
    tfs = tfs | set(table['TF'])
len(tfs)

52

In [19]:
controls = pd.read_table('common/tftypes.txt', names=['tf', 'type'])
controls[controls['tf'].isin(tfs)].groupby('type').count()

,tf
type,
Codebook TF,33
Control,19
